# **Installation of required Libraries**

In [ ]:
!apt-get install poppler-utils
!pip install pdf2image transformers datasets peft accelerate bitsandbytes

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (157 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00


# **Helper Function for Pdf Parsing**

In [ ]:
import os
from pdf2image import convert_from_path
from PIL import Image

def process_pdf_to_images(pdf_path, dpi=200):
    """
    Converts a multi-page PDF into a list of PIL Images.
    Resizes them to prevent VRAM overflow during VLM inference.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found at {pdf_path}")

    print(f"Converting {pdf_path} to images...")
    # Convert PDF pages to images
    pages = convert_from_path(pdf_path, dpi=dpi)

    processed_images = []
    for i, page in enumerate(pages):
        # Ensure RGB mode
        if page.mode != 'RGB':
            page = page.convert('RGB')

        # Resize to max 800px dimension to save VRAM while preserving OCR quality
        w, h = page.size
        max_dim = 800
        if max(w, h) > max_dim:
            ratio = max_dim / float(max(w, h))
            page = page.resize((int(w * ratio), int(h * ratio)), Image.Resampling.LANCZOS)

        processed_images.append(page)
        print(f"  -> Processed Page {i+1}")

    return processed_images

# # Example Usage:
pdf_images = process_pdf_to_images("/content/wordpress-pdf-invoice-plugin-sample.pdf")

Converting /content/wordpress-pdf-invoice-plugin-sample.pdf to images...
  -> Processed Page 1


# **Loading the VLM , and making it ready for the Prompts**

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# 1. Load the Baseline Model (No LoRA, No Fine-tuning!)
model_id = "HuggingFaceTB/SmolVLM-500M-Instruct"

print("Loading VLM Engine...")
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

# 2. Define the Inference Engine
def analyze_document(image, task_type):
    """Routes the image to specific prompts based on the required task."""

    # --- PROMPT ENGINEERING THE TASKS ---
    if task_type == "extraction":
        prompt = (
            "You are a precise data extraction system. Read this receipt and extract the "
            "Total, Tax, and Menu Items. Output the result strictly as a JSON dictionary. "
            "Do not include markdown, code blocks, or conversational text."
        )
    elif task_type == "signature":
        prompt = (
            "Look carefully at this document. Is there a handwritten signature present? "
            "Answer with exactly one word: 'Yes' or 'No'."
        )
    elif task_type == "form_fields":
        prompt = (
            "Identify all form fields, checkboxes, or input lines in this document. "
            "Output a JSON list where each item has the 'field_name' and a 'status' "
            "of either 'filled' or 'empty'."
        )
    else:
        prompt = "Describe this document."

    # --- INFERENCE ---
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False, # STRICT DETEMINISM: Prevents hallucinations
        )

    input_len = inputs["input_ids"].shape[1]
    output_text = processor.batch_decode(generated_ids[:, input_len:], skip_special_tokens=True)[0]

    return output_text.strip()

Loading VLM Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

# **Evaluation on Cord-V2 Test dataset**

In [ ]:
import json
import re
from datasets import load_dataset
from tqdm import tqdm

# --- METRIC UTILITIES ---
def clean_json_string(output_str):
    """Strips markdown and uses regex fallback for VLM syntax errors."""
    cleaned = re.sub(r"```json\s*", "", output_str)
    cleaned = re.sub(r"\s*```", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        rescued_dict = {}
        pattern = r'"([^"]+)"\s*:\s*(?:"([^"]*)"|([A-Za-z0-9\.,]+))'
        for match in re.finditer(pattern, cleaned):
            key = match.group(1)
            value = match.group(2) if match.group(2) is not None else match.group(3)
            rescued_dict[key] = value
        return rescued_dict

def extract_all_kv_pairs(d):
    """Flattens dictionary for strict key-value scoring."""
    items = set()
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                items.update(extract_all_kv_pairs(v))
            elif str(v).strip():
                items.add((str(k).strip().lower(), str(v).strip().lower()))
    elif isinstance(d, list):
        for item in d:
            items.update(extract_all_kv_pairs(item))
    return items

def compute_f1_score(true_dict, pred_dict):
    true_pairs = extract_all_kv_pairs(true_dict)
    pred_pairs = extract_all_kv_pairs(pred_dict)

    true_positives = len(true_pairs.intersection(pred_pairs))
    precision = true_positives / len(pred_pairs) if len(pred_pairs) > 0 else 0.0
    recall = true_positives / len(true_pairs) if len(true_pairs) > 0 else 0.0

    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    acc = true_positives / len(true_pairs.union(pred_pairs)) if len(true_pairs.union(pred_pairs)) > 0 else 0.0
    return acc, f1

# --- EVALUATION PIPELINE ---
print("Loading CORD Dataset for Evaluation...")
dataset = load_dataset("naver-clova-ix/cord-v2")
test_subset = dataset['test'].select(range(10)) # Evaluate on 10 documents

total_acc, total_f1, valid_docs = 0, 0, 0

print("\nRunning Zero-Shot Extraction Pipeline...")
for i, example in enumerate(tqdm(test_subset)):
    # 1. Ground Truth
    raw_gt = json.loads(example['ground_truth'])
    clean_gt = raw_gt.get('gt_parse', {})

    # 2. VLM Inference (Using the Module 2 function)
    # Notice we resize the image directly here if it didn't come from Module 1
    image = example['image'].convert('RGB')
    raw_output = analyze_document(image, task_type="extraction")

    # 3. Parse and Score
    pred_json = clean_json_string(raw_output)

    if not clean_gt:
        continue

    valid_docs += 1
    acc, f1 = compute_f1_score(clean_gt, pred_json)
    total_acc += acc
    total_f1 += f1

    # Print the first few for visual confirmation
    if i < 3:
        print(f"\n--- Document {i+1} ---")
        print(f"Model RAW Output: {raw_output}")
        print(f"F1 Score: {f1:.4f}")

# Final Scores
if valid_docs > 0:
    print("\n" + "="*40)
    print("      ZERO-SHOT EVALUATION METRICS")
    print("="*40)
    print(f"Total Documents    : {valid_docs}")
    print(f"Average Accuracy   : {(total_acc / valid_docs) * 100:.2f}%")
    print(f"Average F1-Score   : {(total_f1 / valid_docs):.4f}")
    print("="*40)

Loading CORD Dataset for Evaluation...


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00004-7dbbe248962764(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

...


--- Document 2 ---
Model RAW Output: {
    "menu": [
        {
            "nm": "J.STB PROMO",
            "cnt": "17500",
            "price": "17500"
        },
        {
            "nm": "Y.B.B.BAT",
            "cnt": "46000",
            "price": "46000"
        },
        {
            "nm": "Y.BASO PROM",
            "cnt": "27500",
            "price": "27500"
        }
    ],
    "total": {
        "total_price": "91000",
        "cashprice": "91000"
    }
}
F1 Score: 0.7368


 30%|███       | 3/10 [00:27<01:03,  9.01s/it]


--- Document 3 ---
Model RAW Output: {
    "menu": [
        {
            "nm": "MSMINT MT (L)",
            "cnt": "1",
            "price": "24.000"
        },
        {
            "nm": "COCONUT JELLY (L)",
            "cnt": "1",
            "price": "4,000"
        }
    ],
    "sub_total": {
        "subtotal_price": "28.000",
        "tax_price": "1"
    },
    "total": {
        "total_price": "28.000",
        "cashprice": "100.000",
        "changeprice": "72.000",
        "menuqty_cnt": "1"
    }
}
F1 Score: 0.2857


100%|██████████| 10/10 [01:26<00:00,  8.64s/it]


      ZERO-SHOT EVALUATION METRICS
Total Documents    : 10
Average Accuracy   : 38.07%
Average F1-Score   : 0.5101


# **Visualization on a Sample Invoice PDF**

In [ ]:
# Path to your local sample PDF
pdf_path = "/content/wordpress-pdf-invoice-plugin-sample.pdf"

# 1. Execute Module 1: Convert PDF to Images
# Note: Assignment requires processing each page separately
pages = process_pdf_to_images(pdf_path)

# List of tasks defined in Module 2
tasks = ["extraction", "signature", "form_fields"]

# 2. Loop through every page and every task
all_results = {}

print(f"\nStarting End-to-End Analysis for: {pdf_path}")
for i, page_image in enumerate(pages):
    page_key = f"page_{i+1}"
    all_results[page_key] = {}

    print(f"\n" + "="*30)
    print(f" ANALYZING PAGE {i+1}")
    print("="*30)

    for task in tasks:
        print(f"Running Task: [{task}]...")

        # Execute Module 2 Inference
        raw_result = analyze_document(page_image, task_type=task)

        # If the task is JSON-based, clean and parse it
        if task in ["extraction", "form_fields"]:
            parsed_result = clean_json_string(raw_result)
            all_results[page_key][task] = parsed_result
        else:
            # Signature detection is a binary string
            all_results[page_key][task] = raw_result

        print(f"  Result: {all_results[page_key][task]}")

# 3. Final Output: Module 3 Structured JSON
print("\n" + "#"*50)
print(" FINAL STRUCTURED OUTPUT (Module 3 Deliverable)")
print("#"*50)
print(json.dumps(all_results, indent=2))

# Save results to a JSON file for your assignment submission
with open("extraction_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

Converting /content/wordpress-pdf-invoice-plugin-sample.pdf to images...
  -> Processed Page 1

Starting End-to-End Analysis for: /content/wordpress-pdf-invoice-plugin-sample.pdf

 ANALYZING PAGE 1
Running Task: [extraction]...
  Result: {'Invoice': {'No': 'INV-3337', 'Order': {'No': '12345', 'City': 'Your City', 'Zip': '12345', 'Address': {'Street': 'Suite 5A-1204', 'City': 'Your City', 'State': 'VA', 'Zip': '12345'}}, 'Test Business': {'Address': {'Street': '123 Somewhere St', 'City': 'Melbourne', 'Zip': '3000'}}}, 'Hrs/Qty': {'Web Design': '1.00', 'Service': '1.00'}, 'Rate/Price': [{'Rate': '85.00', 'Price': '0.00%'}, {'Rate': '85.00', 'Price': '0.00%'}], 'Sub Total': {'Subtotal': '85.00'}, 'Tax': {'Tax_price': '8.50'}, 'Total': {'Total_price': '93.50', 'Payment_price': '93.50'}}
Running Task: [signature]...
  Result: Yes.
Running Task: [form_fields]...
  Result: {'header': {'title': 'Invoice', 'subheader': 'From: DEMO - Sliced Invoices', 'subsubheader': 'Suite 5A-1204', 'subsubsubh